# 06 — Feature Selection

## Objective

This notebook reduces the 229-feature engineered set from Notebook 05 by selecting the subset of features that carries the most predictive signal, using three complementary methods:

1. **Filter method** — Mutual Information between each feature and the target, independent of any model.
2. **Embedded method** — Feature importances from a fitted CatBoost model.
3. **Wrapper method** — Recursive Feature Elimination (RFE), using a lightweight estimator for ranking efficiency.

Each method is evaluated at multiple feature-count thresholds on the validation set, using the two champion models (CatBoost, LightGBM). The goal is to find a feature subset that matches or improves on the 229-feature ROC-AUC from Notebook 05 while further reducing dimensionality.

### Principles

- `random_state = 42`
- All feature ranking is computed from the training set only.
- The test set is not touched in this notebook.
- Results from every method/threshold combination are logged to `reports/` for transparent comparison.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Load the engineered, preprocessed data from Notebook 05
fe_data = joblib.load(MODELS_DIR / "fe_data.pkl")

X_train_processed = fe_data["X_train"]
X_val_processed = fe_data["X_val"]
X_test_processed = fe_data["X_test"]

y_train = fe_data["y_train"]
y_val = fe_data["y_val"]
y_test = fe_data["y_test"]

# Load the preprocessor to recover feature names after one-hot encoding
fe_preprocessor = joblib.load(MODELS_DIR / "fe_preprocessor.pkl")
feature_names = fe_preprocessor.get_feature_names_out()

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)
print("\nNumber of feature names:", len(feature_names))
print("Feature names match processed columns:", len(feature_names) == X_train_processed.shape[1])

Train: (63444, 229)
Validation: (15775, 229)
Test: (20124, 229)

Number of feature names: 229
Feature names match processed columns: True


## Method 1 — Filter: Mutual Information

Mutual Information (MI) measures the statistical dependency between each feature and the target, without involving any specific model. It is computed once on the training set and used to rank all 229 features by relevance.

This is the cheapest and most model-agnostic of the three methods — a useful first pass to identify features with essentially no relationship to the target (candidates for removal regardless of which model is used downstream).

In [2]:
from sklearn.feature_selection import mutual_info_classif

# Convert sparse matrix to dense for mutual_info_classif (229 features is small enough)
X_train_dense = X_train_processed.toarray() if hasattr(X_train_processed, "toarray") else X_train_processed

mi_scores = mutual_info_classif(
    X_train_dense, y_train, random_state=RANDOM_STATE, discrete_features=False
)

mi_ranking = pd.DataFrame({
    "feature": feature_names,
    "mi_score": mi_scores
}).sort_values("mi_score", ascending=False).reset_index(drop=True)

print("Top 20 features by Mutual Information:")
print(mi_ranking.head(20).to_string(index=False))

print("\nBottom 20 features by Mutual Information:")
print(mi_ranking.tail(20).to_string(index=False))

print(f"\nFeatures with MI score == 0: {(mi_ranking['mi_score'] == 0).sum()}")

Top 20 features by Mutual Information:
                                        feature  mi_score
               numeric__total_prior_utilization  0.036445
                      numeric__number_inpatient  0.034128
                      numeric__number_diagnoses  0.012419
                    categorical__tolbutamide_No  0.012243
                     numeric__number_outpatient  0.011821
                 categorical__chlorpropamide_No  0.010675
                    categorical__repaglinide_No  0.010200
                      numeric__number_emergency  0.010148
                    categorical__nateglinide_No  0.009926
                   categorical__diabetesMed_Yes  0.009647
categorical__medical_specialty_InternalMedicine  0.009067
                   categorical__troglitazone_No  0.008184
                    categorical__glimepiride_No  0.008050
                       categorical__acarbose_No  0.007307
                     categorical__tolazamide_No  0.007217
                     numeric__adm

## Method 2 — Embedded: CatBoost Feature Importance

A CatBoost model is fit on the full 229-feature training set, and its built-in feature importances (based on how much each feature contributes to reducing loss across all trees) are used to rank features.

Unlike Mutual Information, this ranking reflects feature usefulness *specifically for CatBoost's decision-tree structure*, capturing interactions between features that a univariate filter method cannot.

In [3]:
from catboost import CatBoostClassifier

cb_for_importance = CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)
cb_for_importance.fit(X_train_processed, y_train)

importance_ranking = pd.DataFrame({
    "feature": feature_names,
    "importance": cb_for_importance.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("Top 20 features by CatBoost importance:")
print(importance_ranking.head(20).to_string(index=False))

print("\nBottom 20 features by CatBoost importance:")
print(importance_ranking.tail(20).to_string(index=False))

print(f"\nFeatures with importance == 0: {(importance_ranking['importance'] == 0).sum()}")

Top 20 features by CatBoost importance:
                              feature  importance
     numeric__total_prior_utilization    6.556042
            numeric__number_inpatient    6.320955
         numeric__admission_source_id    5.344416
            numeric__number_diagnoses    4.961627
    numeric__discharge_disposition_id    4.715999
             numeric__num_medications    4.690774
         numeric__medications_per_day    4.347971
            numeric__total_procedures    3.979961
      numeric__lab_procedures_per_day    3.951928
          numeric__num_lab_procedures    3.644263
              numeric__num_procedures    3.088193
           numeric__admission_type_id    2.911402
            numeric__time_in_hospital    2.057245
           numeric__number_outpatient    1.794686
            numeric__number_emergency    1.481431
  numeric__num_medications_prescribed    1.365061
           categorical__payer_code_MC    1.016257
             categorical__age_[80-90)    0.958130
categorica

## Method 3 — Wrapper: Recursive Feature Elimination (RFE)

RFE repeatedly fits a model, ranks features by importance/coefficient, removes the weakest, and refits — capturing interactions between remaining features at each step, unlike the one-shot filter and embedded methods above.

**First attempt (Logistic Regression) — rejected.** RFE was initially run with a Logistic Regression estimator. The resulting ranking was inconsistent with both the Mutual Information and CatBoost importance rankings: strong, well-established predictors like `number_inpatient` and `discharge_disposition_id` ranked near the bottom, while rare one-hot dummy columns (e.g. `medical_specialty_Pediatrics-Endocrinology`, present for only a handful of patients) ranked at the very top. This is a known failure mode — rare categorical dummies can produce inflated logistic regression coefficients due to quasi-separation, even though they carry little genuine signal. Since RFE ranks by coefficient magnitude, it was misled by these artificially large coefficients.

**Final approach — Decision Tree estimator.** RFE was re-run using a shallow Decision Tree (`max_depth=8`, `min_samples_leaf=50`) as the estimator. The minimum leaf size makes tree splits robust to rare categories, since a split cannot isolate a handful of observations. This produced a ranking consistent with the filter and embedded methods above, and is used for the rest of this notebook.

This RFE ranking is a purely rank-based signal; the actual feature subset selection is validated using the real champion models (CatBoost, LightGBM) in the evaluation step below, so the final decision is not biased by the ranking estimator's own limitations.

In [5]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

# A shallow tree with a minimum leaf size is much more robust to rare
# one-hot dummy columns than Logistic Regression coefficients, which can be
# distorted by quasi-separation on sparse rare categories.
rfe_estimator = DecisionTreeClassifier(
    random_state=RANDOM_STATE, max_depth=8, min_samples_leaf=50
)

rfe = RFE(estimator=rfe_estimator, n_features_to_select=1, step=10)
rfe.fit(X_train_processed, y_train)

rfe_ranking = pd.DataFrame({
    "feature": feature_names,
    "rfe_rank": rfe.ranking_
}).sort_values("rfe_rank").reset_index(drop=True)

print("Top 20 features by RFE ranking (Decision Tree estimator):")
print(rfe_ranking.head(20).to_string(index=False))

print("\nBottom 20 features by RFE ranking:")
print(rfe_ranking.tail(20).to_string(index=False))

Top 20 features by RFE ranking (Decision Tree estimator):
                            feature  rfe_rank
   numeric__total_prior_utilization         1
  numeric__discharge_disposition_id         2
       numeric__admission_source_id         2
        numeric__num_lab_procedures         2
           numeric__num_medications         2
numeric__num_medications_prescribed         2
          numeric__number_inpatient         2
          numeric__number_diagnoses         2
       numeric__medications_per_day         2
         numeric__admission_type_id         3
           categorical__age_[80-90)         3
           categorical__age_[70-80)         3
          categorical__insulin_Down         3
          numeric__total_procedures         3
         categorical__payer_code_MC         3
          numeric__number_emergency         3
          numeric__time_in_hospital         3
    numeric__lab_procedures_per_day         3
            numeric__num_procedures         3
          categorical_

## Combine Rankings & Select Feature Subsets

The three rankings (Mutual Information, CatBoost importance, RFE) are combined into a single reference table. Each method's top-K features are then evaluated at several thresholds (K = 50, 100, 150) using both champion models on the validation set, to find the smallest feature subset that matches or exceeds the 229-feature performance from Notebook 05.

In [6]:
combined_ranking = (
    mi_ranking[["feature", "mi_score"]]
    .merge(importance_ranking[["feature", "importance"]], on="feature")
    .merge(rfe_ranking[["feature", "rfe_rank"]], on="feature")
)

# Normalized rank per method (1 = best) for a simple average-rank comparison
combined_ranking["mi_rank"] = combined_ranking["mi_score"].rank(ascending=False, method="min")
combined_ranking["importance_rank"] = combined_ranking["importance"].rank(ascending=False, method="min")
combined_ranking["avg_rank"] = combined_ranking[["mi_rank", "importance_rank", "rfe_rank"]].mean(axis=1)

combined_ranking = combined_ranking.sort_values("avg_rank").reset_index(drop=True)

print("Top 20 features by average rank across all three methods:")
print(combined_ranking[["feature", "mi_rank", "importance_rank", "rfe_rank", "avg_rank"]].head(20).to_string(index=False))

combined_ranking.to_csv(REPORTS_DIR / "feature_selection_rankings.csv", index=False)
print(f"\nFull ranking table saved to: {REPORTS_DIR / 'feature_selection_rankings.csv'}")

Top 20 features by average rank across all three methods:
                                        feature  mi_rank  importance_rank  rfe_rank  avg_rank
               numeric__total_prior_utilization      1.0              1.0         1  1.000000
                      numeric__number_inpatient      2.0              2.0         2  2.000000
                      numeric__number_diagnoses      3.0              4.0         2  3.000000
                     numeric__number_outpatient      5.0             14.0         4  7.666667
                      numeric__number_emergency      8.0             15.0         3  8.666667
                     numeric__admission_type_id     16.0             12.0         3 10.333333
                   numeric__admission_source_id     28.0              3.0         2 11.000000
              numeric__discharge_disposition_id     37.0              5.0         2 14.666667
                    categorical__race_Caucasian     25.0             22.0         5 17.333333
  

## Evaluate Feature Subsets on Validation Set

The combined ranking is used to select the top-K features at several thresholds (K = 30, 50, 100, 150), and the champion models (CatBoost, LightGBM) are retrained on each subset. This directly measures whether a smaller, ranked subset can match or exceed the full 229-feature ROC-AUC from Notebook 05 — the goal is the smallest subset with no meaningful performance loss.

In [7]:
from sklearn.metrics import roc_auc_score
import time

# Map feature names to column positions in the processed (dense/sparse) matrix
feature_to_index = {name: idx for idx, name in enumerate(feature_names)}

thresholds = [30, 50, 100, 150, 229]  # 229 = full set, as a sanity-check anchor

selection_results = []

for k in thresholds:
    top_k_features = combined_ranking["feature"].head(k).tolist()
    col_indices = [feature_to_index[f] for f in top_k_features]

    X_train_subset = X_train_processed[:, col_indices]
    X_val_subset = X_val_processed[:, col_indices]

    for model_name, model_class, model_kwargs in [
        ("CatBoost", __import__("catboost").CatBoostClassifier, {"random_state": RANDOM_STATE, "verbose": 0}),
        ("LightGBM", __import__("lightgbm").LGBMClassifier, {"random_state": RANDOM_STATE, "n_jobs": -1, "verbose": -1}),
    ]:
        model = model_class(**model_kwargs)
        start = time.time()
        model.fit(X_train_subset, y_train)
        train_time = time.time() - start

        y_prob = model.predict_proba(X_val_subset)[:, 1]
        auc = roc_auc_score(y_val, y_prob)

        selection_results.append({
            "k_features": k,
            "Model": model_name,
            "ROC-AUC": auc,
            "Train Time (s)": round(train_time, 2),
        })
        print(f"k={k:>3} | {model_name:<10} | ROC-AUC = {auc:.4f} | time = {train_time:.2f}s")

selection_results_df = pd.DataFrame(selection_results)

k= 30 | CatBoost   | ROC-AUC = 0.6798 | time = 17.18s
k= 30 | LightGBM   | ROC-AUC = 0.6797 | time = 0.55s
k= 50 | CatBoost   | ROC-AUC = 0.6820 | time = 16.69s
k= 50 | LightGBM   | ROC-AUC = 0.6801 | time = 0.59s
k=100 | CatBoost   | ROC-AUC = 0.6854 | time = 18.38s
k=100 | LightGBM   | ROC-AUC = 0.6836 | time = 0.62s
k=150 | CatBoost   | ROC-AUC = 0.6876 | time = 21.46s
k=150 | LightGBM   | ROC-AUC = 0.6835 | time = 0.80s
k=229 | CatBoost   | ROC-AUC = 0.6859 | time = 18.02s
k=229 | LightGBM   | ROC-AUC = 0.6835 | time = 0.65s


## Select Final Feature Subset

k = 150 achieves the best CatBoost ROC-AUC (0.6876) across all tested thresholds — matching or slightly exceeding the full 229-feature set while using ~35% fewer features. LightGBM performance plateaus from k=100 onward, so it does not further constrain the choice.

**k = 150 is selected as the final feature set** going into the optimization stage (Notebook 07).

In [8]:
FINAL_K = 150

final_features = combined_ranking["feature"].head(FINAL_K).tolist()
final_col_indices = [feature_to_index[f] for f in final_features]

X_train_selected = X_train_processed[:, final_col_indices]
X_val_selected = X_val_processed[:, final_col_indices]
X_test_selected = X_test_processed[:, final_col_indices]

print(f"Selected {len(final_features)} features.")
print("Shapes after selection:")
print("Train:", X_train_selected.shape)
print("Validation:", X_val_selected.shape)
print("Test:", X_test_selected.shape)

# Save selected feature set for reuse in Notebook 07
selected_data = {
    "X_train": X_train_selected, "X_val": X_val_selected, "X_test": X_test_selected,
    "y_train": y_train, "y_val": y_val, "y_test": y_test,
    "feature_names": final_features,
}
joblib.dump(selected_data, MODELS_DIR / "selected_data.pkl")
joblib.dump(final_col_indices, MODELS_DIR / "selected_feature_indices.pkl")

print(f"\nSelected data saved to: {MODELS_DIR / 'selected_data.pkl'}")

# Log final selection results to the running comparison log
selection_results_df.to_csv(REPORTS_DIR / "feature_selection_results.csv", index=False)

comparison_log_path = REPORTS_DIR / "model_comparison_log.csv"
existing_log = pd.read_csv(comparison_log_path)

final_k_results = selection_results_df[selection_results_df["k_features"] == FINAL_K].copy()
log_entry = final_k_results[["Model", "ROC-AUC", "Train Time (s)"]].copy()
log_entry.insert(0, "stage", f"03_feature_selection_k{FINAL_K}")
log_entry.insert(1, "notebook", "06_feature_selection")

updated_log = pd.concat([existing_log, log_entry], ignore_index=True)
updated_log.to_csv(comparison_log_path, index=False)

print(f"\nComparison log updated with stage: 03_feature_selection_k{FINAL_K}")

Selected 150 features.
Shapes after selection:
Train: (63444, 150)
Validation: (15775, 150)
Test: (20124, 150)

Selected data saved to: e:\all projects\Machine Learning\Advanced-ML-Hospital-Readmission\models\selected_data.pkl

Comparison log updated with stage: 03_feature_selection_k150


## Three-Stage Comparison & Summary

In [9]:
comparison_all = updated_log[
    (updated_log["stage"].isin(["01_baseline", "02_feature_engineering", "03_feature_selection_k150"])) &
    (updated_log["Model"].isin(["CatBoost", "LightGBM"]))
].pivot(index="Model", columns="stage", values="ROC-AUC")

comparison_all = comparison_all[
    ["01_baseline", "02_feature_engineering", "03_feature_selection_k150"]
]

print("="*70)
print("ROC-AUC ACROSS ALL STAGES")
print("="*70)
print(comparison_all.round(4))

feature_counts = pd.Series({
    "01_baseline": 2304,
    "02_feature_engineering": 229,
    "03_feature_selection_k150": 150,
}, name="n_features")

print("\nFeature count per stage:")
print(feature_counts)

ROC-AUC ACROSS ALL STAGES
stage     01_baseline  02_feature_engineering  03_feature_selection_k150
Model                                                                   
CatBoost       0.6918                  0.6870                     0.6876
LightGBM       0.6889                  0.6835                     0.6835

Feature count per stage:
01_baseline                  2304
02_feature_engineering        229
03_feature_selection_k150     150
Name: n_features, dtype: int64


## Summary

Three feature selection methods were applied to the 229-feature engineered set from Notebook 05:

1. **Filter (Mutual Information)** — model-agnostic univariate ranking.
2. **Embedded (CatBoost importance)** — ranking from a fitted CatBoost model.
3. **Wrapper (RFE)** — initially attempted with Logistic Regression, which produced an unreliable ranking due to quasi-separation on rare one-hot categories (established predictors like `number_inpatient` ranked near the bottom). Re-run with a Decision Tree estimator (`max_depth=8`, `min_samples_leaf=50`), which produced a ranking consistent with the other two methods.

The three rankings were combined via average rank, and evaluated at multiple thresholds (k = 30, 50, 100, 150, 229) using the champion models on the validation set.

**k = 150 features was selected as the final set**, achieving CatBoost ROC-AUC = 0.6876 — matching or slightly exceeding the full 229-feature performance while using ~35% fewer features. LightGBM's performance plateaued from k = 100 onward.

### Results Across All Stages (CatBoost)

| Stage | Features | ROC-AUC |
|---|---|---|
| Baseline | 2,304 | 0.6918 |
| Feature Engineered | 229 | 0.6870 |
| Feature Selected (k=150) | 150 | 0.6876 |

The feature selection stage recovered most of the small ROC-AUC drop introduced by feature engineering, while achieving a combined ~93% reduction in feature dimensionality relative to the original baseline (2,304 → 150).

Results are logged in `reports/feature_selection_rankings.csv`, `reports/feature_selection_results.csv`, and appended to `reports/model_comparison_log.csv` under `stage = 03_feature_selection_k150`.

**Next notebook (07):** Hyperparameter optimization — implementing and comparing multiple optimization algorithms (Grid/Random Search as classical baselines, Bayesian optimization, and a Genetic Algorithm) on the two champion models using this 150-feature set.